In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

from fraud_engine.data.load import DEFAULT_CONFIG_PATH, load_config

In [ ]:
# A notebook's cwd is unreliable - anchor to the repo root instead.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

config = load_config(ROOT / DEFAULT_CONFIG_PATH)

# config.yaml paths are repo-root-relative.
interim_path = ROOT / config["paths"]["interim"]

In [ ]:
# Hold back the tail: Phase 02 carves the test set from it, and a boundary chosen
# after seeing its fraud rate is contaminated. Phase 02 sets the real number.
EDA_MAX_DAY = 120
EDA_FILTER = [("day", "<=", EDA_MAX_DAY)]


# One place enforces the horizon - a bare read_parquet later would span all 182 days.
def read_eda(columns):
    return pd.read_parquet(interim_path, columns=columns, filters=EDA_FILTER)

### Section 1 - base rate and time span

How much fraud there is, and over how long — measured inside the EDA horizon, not
across the full file. The base rate fixes the class imbalance every later design
decision has to survive, and is the reason accuracy is never reported here.

In [ ]:
df = read_eda(["day", "isFraud", "TransactionAmt"])

row_count = len(df)
print(f"Row count: {row_count}")

day_span = {
    "first_day": int(df["day"].min()),
    "last_day": int(df["day"].max()),
    "distinct_day_count": df["day"].nunique(),
}
print(f"Day span: {day_span}")

fraud_count = (df["isFraud"] == 1).sum()
print(f"Fraud count: {fraud_count}")

fraud_rate = fraud_count / row_count
print(f"Fraud rate: {fraud_rate:.3%}")

#### What the numbers say

414,542 transactions over 120 days (days 1–120 of the file's 182), no missing days.
14,600 of them are fraud — a base rate of **3.522%**.

Two consequences follow directly from that rate.

**Accuracy is unusable.** A model that predicts "not fraud" for every row scores
**96.478%**. Any accuracy figure quoted for this problem is describing the class balance,
not the model, which is why PR-AUC and recall@capacity are the metrics here.

**Manual review cannot be the answer on its own.** At ~3,455 transactions/day and the
`review_capacity: 0.01` committed in `cost_matrix.yaml`, the queue holds ~35 reviews/day
against ~122 frauds/day. Even a *perfect* ranker that spent every review slot on a true
fraud would top out at **28.4% recall**. The remaining ~72% has to be handled by the
allow/block decision, which is the constraint the Phase 06 cost policy exists to resolve.

`day` is an offset from an unpublished reference point, so this is a duration, not a date
range — no calendar claims can be made from it.

### Section 2 - fraud rate over time

Daily fraud rate across the horizon, with a 7-day centred rolling mean over the raw
series and transaction volume on the panel beneath — a rate move that coincides with
a volume move has a different explanation from one that does not.

The figure is saved to `reports/figures/`. Whether the
rate holds steady or drifts is the evidence the Phase 02 temporal split rests on.

In [ ]:
daily = df.groupby("day").agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
print(daily.shape)

In [ ]:
# The rate is only interpretable next to the volume that produced it.
fig, (ax_rate, ax_vol) = plt.subplots(2, 1, sharex=True, figsize=(11, 6), height_ratios=[2, 1])

# At ~3,000 transactions/day the raw series is mostly binomial noise.
ax_rate.plot(daily.index, daily["rate"], lw=1, alpha=0.35, label="daily")
ax_rate.plot(
    daily.index,
    daily["rate"].rolling(7, center=True).mean(),
    lw=2,
    label="7-day rolling mean (centred)",
)

# From zero: autoscale would amplify the noise into a trend.
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
ax_rate.set_title(f"Fraud rate over time (days 1-{EDA_MAX_DAY})")
ax_rate.legend(loc="upper left", frameon=False)
ax_rate.grid(alpha=0.25)

ax_vol.fill_between(daily.index, daily["volume"], alpha=0.4, lw=0)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
# `day` is an offset from an unpublished reference, not a calendar date.
ax_vol.set_xlabel("day (relative to an unpublished reference, not a calendar date)")
ax_vol.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

#### What the chart shows

The fraud rate is **not stationary** across the window.

Days 1–22 show volume climbing 55% while the daily fraud *count* stays flat (−3%), so the
rate falls to a 1.83% trough purely by dilution — the surge is legitimate customers, not
quieter attackers. Volume peaks and the rate bottoms on the same day (22), giving a
smoothed correlation of −0.88 over this stretch.

After that the series steps to a new level rather than continuing to trend. The 30-day
means for days 31–60, 61–90 and 91–120 are 4.03%, 4.01% and 4.02% — but those flat
averages hide real movement inside them. Days 91–120 rise from 3.41% to 4.79%, driven by a
20% increase in daily fraud count against a 25% fall in legitimate volume. That rise is
*not* dilution: it is more fraud against a smaller base.

The 30-day window width is a choice, and the flatness of those means is partly an artifact
of it.

**Implication for Phase 02.** A random split would scatter both regimes across train and
test, letting the model learn from a fraud environment it would not have had in
production. The split must be temporal.

### Section 3 - hour alignment and time of day

`TransactionDT` counts seconds from a reference point Vesta never published, so `hour`
(`seconds // 3600 % 24`) is a consistent 24-hour cycle but not necessarily wall-clock time.
Bucket 0 is midnight only if that reference is midnight-aligned — `min(TransactionDT)` is
exactly 86,400, one whole day, which hints at it but is an inference, not a fact.

**The volume curve is the test.** Human commerce has a deep overnight trough. If one appears
at plausible night hours, `hour` means hour-of-day and time-of-day claims are legitimate. If
the curve is flat or oddly phased, `hour` stays a usable cyclic feature but no time-of-day
claim can be made from it — and the verdict goes back into the `add_time_columns` docstring.

In [ ]:
hourly = (
    read_eda(["hour", "isFraud"])
    .groupby("hour")
    .agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
)
print(hourly.shape)

In [ ]:
# Volume on top: this curve is the alignment test, the rate is only interpretable once it
# is settled. No smoothing - each bucket holds ~19,000 transactions (noise is +/-0.15pp),
# and rolling() would treat hours 0 and 23 as distant rather than adjacent, blanking
# exactly the midnight window being tested.
fig, (ax_vol, ax_rate) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))

ax_vol.bar(hourly.index, hourly["volume"], width=0.85, alpha=0.7)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
ax_vol.set_title(f"Volume and fraud rate by hour bucket (days 1-{EDA_MAX_DAY})")
ax_vol.grid(alpha=0.25)

ax_rate.plot(hourly.index, hourly["rate"], lw=2, marker="o", ms=4)
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
# Bucket 0 is midnight only if the unpublished reference is midnight-aligned.
ax_rate.set_xlabel("hour bucket (0-23; bucket 0 is not necessarily midnight)")
ax_rate.set_xticks(range(0, 24, 2))
ax_rate.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

#### What the chart shows

**Bucket 0 is not midnight.** Volume swings 19× between bucket 9 (1,580) and bucket 19
(30,242) — an unmistakable diurnal cycle, but phased wrong for bucket 0 to be midnight,
which would mean the fewest transactions at 9am and the most between 7pm and 1am.

Placing the trough at a plausible pre-dawn hour puts midnight at **bucket 4–6**. Past
bucket 6 the mapping breaks: at bucket 8 the daily low lands at 1am, at bucket 12 at 9pm.
An offset of roughly +5 hours matches US Eastern's offset from UTC, which would be the
expected artifact of UTC timestamps against a mostly-US customer base — a hypothesis with
a mechanism, not a documented fact.

Consequence: `hour` is a **cyclic feature, not a wall-clock label**, and no time-of-day
claim can be made without carrying that offset as a stated assumption. The verdict is
recorded in the `add_time_columns` docstring, which previously left the question open.

#### The rate peak is a denominator effect

The fraud rate peaks at bucket 8 (10.01%, against 3.45% across buckets 17–23), and that
peak sits exactly where volume is lowest. But the counts run the other way:

| | frauds/day | legit/day | rate |
|---|---|---|---|
| bucket 8 | 1.6 | 15 | 10.01% |
| buckets 17–23 | 8.5 | 237 | 3.45% |

Relative to the plateau, fraud falls to 0.19× while legitimate activity falls to 0.06×.
Both collapse overnight; legitimate activity collapses harder. **Fraud does not peak there
— it declines more slowly than everything around it.** The hours with the most fraud by
count are the plateau hours, which carry roughly five times more fraud per day.

This is section 2's mechanism inverted: there, a surge in legitimate volume diluted a flat
fraud count; here, a collapse in legitimate volume concentrates a falling one. Rate and
count answer different questions, and only the count says where the money is.

### Section 4 - amount distribution and USD exposure

The headline result of this project is denominated in USD, so this section establishes
the denominator: how much money moves, how much of it is fraud, and whether that loss
sits in a few large transactions or spreads across many.

Two comparisons carry it — fraud against legitimate at each percentile, and fraud's
share of USD against its share of transaction count.

In [ ]:
amounts_df = read_eda(["TransactionAmt", "isFraud"])

qs = [0.25, 0.5, 0.75, 0.9, 0.99]

amounts_df.groupby("isFraud")["TransactionAmt"].describe(percentiles=qs)

In [ ]:
all_amounts = amounts_df["TransactionAmt"]
fraud_amounts = amounts_df.loc[amounts_df["isFraud"] == 1, "TransactionAmt"]
legit_amounts = amounts_df.loc[amounts_df["isFraud"] == 0, "TransactionAmt"]

total_amount = all_amounts.sum()
total_fraud_amount = fraud_amounts.sum()

print(f"Total amount:       {total_amount:>13,.2f} USD")
print(f"Total fraud amount: {total_fraud_amount:>13,.2f} USD")

# Fraud takes a bigger bite of the money than of the count, by roughly the gap
# between the two class means.
print(f"\nfraud share of USD:   {total_fraud_amount / total_amount:.3%}")
print(f"fraud share of count: {len(fraud_amounts) / len(all_amounts):.3%}")


def top_share(values, fraction):
    """Share of total USD held by the largest `fraction` of transactions."""
    return values.nlargest(int(fraction * len(values))).sum() / values.sum()


# Concentration means nothing without a baseline - every heavy-tailed distribution
# looks concentrated. The question is whether fraud USD is more or less concentrated
# than ordinary commerce.
print("\n         fraud   legit")
for fraction in (0.01, 0.05, 0.10):
    print(
        f"top {fraction:>4.0%}  {top_share(fraud_amounts, fraction):>6.2%}  "
        f"{top_share(legit_amounts, fraction):>6.2%}"
    )

# The README's headline unit. Gross exposure, nothing intercepting it - every later
# number gets measured against this one.
usd_per_1000 = total_fraud_amount / len(all_amounts) * 1000
print(f"\nGross exposure: {usd_per_1000:,.2f} USD per 1,000 transactions")

In [ ]:
# Log-spaced bins with a log x-axis, so ticks read in dollars rather than powers of
# ten. Shared bins across both classes, or the shapes are not comparable.
bins = np.logspace(np.log10(all_amounts.min()), np.log10(all_amounts.max()), 60)

fig, ax = plt.subplots(figsize=(11, 5))

# Weighted to each class's own size rather than density=True: these bins are ~1000x
# wider at the right end, and density divides by bin width, so on a log axis that
# draws them equally wide the bars would mislead. Each bar here is "share of class".
for amounts, label in ((legit_amounts, "legitimate"), (fraud_amounts, "fraud")):
    ax.hist(
        amounts,
        bins=bins,
        weights=np.ones(len(amounts)) / len(amounts),
        histtype="step",
        lw=2,
        label=label,
    )

ax.set_xscale("log")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set_xlabel("transaction amount (USD, log scale)")
ax.set_ylabel("share of class")
ax.set_title(f"Amount distribution by class (days 1-{EDA_MAX_DAY})")
ax.legend(frameon=False)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(ROOT / "reports/figures/amount_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Log-spaced bins smooth over spikes at round values, so the histogram above cannot
# answer this. Compare the share each exact amount holds within its own class: the
# ratio column is how over- or under-represented an amount is among frauds.
top_fraud_amounts = fraud_amounts.value_counts(normalize=True).head(15)

round_numbers = pd.DataFrame(
    {
        "fraud_share": top_fraud_amounts,
        "legit_share": legit_amounts.value_counts(normalize=True).reindex(top_fraud_amounts.index),
    }
)
round_numbers["ratio"] = round_numbers["fraud_share"] / round_numbers["legit_share"]
round_numbers.index.name = "amount"
round_numbers

#### Amounts are compressed, not shifted

Fraud sits **above** legitimate amounts from roughly the 30th to the 98th percentile and
**below** it outside that band in both directions:

| percentile | legit | fraud |
|---|---:|---:|
| 25% | 43.95 | 34.92 |
| 50% | 68.95 | **76.02** |
| 75% | 125.00 | **171.00** |
| 97.5% | 640.95 | **744.95** |
| 99% | 1104.00 | 994.00 |

Fraud is also *less* variable (std 217.5 vs 239.1) despite a higher mean ($146.15 vs
$134.28, +8.8%, t = 6.5) and median (+10.3%). The shape is a compression toward the
middle, not an upward shift — recorded as **H1** in `docs/hypotheses.md`.

The difference is statistically unambiguous but economically modest. Amount alone is a
weak separator; what is interesting is the *shape* of its relationship to fraud.

#### The money is spread, not concentrated

Fraud takes **3.821% of USD** against **3.522% of transactions** — a ratio of 1.085,
matching the gap between the two class means. Amount carries a little information about
exposure, but only a little.

Concentration, against legitimate activity as the baseline:

| top | fraud | legit |
|---|---:|---:|
| 1% | 10.25% | 13.75% |
| 5% | 29.27% | 32.64% |
| 10% | 43.28% | 45.12% |

Fraud USD is **less** concentrated than legitimate USD at every cut. Without the right
column this looks like a concentration finding; against the baseline it is slightly
flatter than ordinary commerce.

This matters for Phase 06. Had losses been concentrated in a handful of large
transactions, catching those few would dominate the result and overall recall would be a
poor proxy for money saved. They are not — **broad recall is what pays here.**

**Gross exposure: 5,147.45 USD per 1,000 transactions** (~17,782 USD/day). Nothing
intercepting it; every later number is measured against this.

#### Round amounts, but only in a band

Round $50 multiples between $150 and $500 are over-represented among frauds — $300 at
4.74×, $450 at 12.94×, $150 at 2.08×. But **$100 runs the other way at 0.68×** despite
being 4.01% of all transactions, and everything above $500 sits at parity. The general
"fraudsters like round numbers" claim does not hold; the banded version does. Recorded
as **H2**.

> **Methodological note.** The histogram cannot show this. Its tall spike near $100 is a
> $21.80-wide bin holding 65,813 transactions across 651 distinct amounts, whose largest
> contributors are $117.00, $100.00 and $107.95. Reading that spike as round-number
> behaviour would invert the actual finding. Binning destroys exactly the information
> the question is about — the `value_counts` comparison is the evidence, not the chart.